# Dataset for evaluation / ds_ct1_20x_224_o112_1

In [1]:
dataset_id = "ds_ct1_20x_224_o112"
training_dataset_id = "ds_ct1_20x_224_o112_1"

## Utils

In [2]:
import itertools
import pandas as pd
import re
import os
import sys
from pathlib import Path

# Add the project root to sys.path so that 'src' can be imported
project_root = Path.cwd().parents[4]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.constants import get_full_path

In [3]:
n_new_colors = {
    0: "#e8f5e9",
    1: "#c8e6c9",
    2: "#a5d6a7",
    3: "#81c784",
    4: "#66bb6a",
    5: "#43a047",
}

def summarize_combinations(
    meta_df: pd.DataFrame,
    cols_to_use,
    cols,
    training,
    new
):
    """
    meta_df     : dataframe with metadata columns and an 'slide_id' column
    cols_to_use : iterable of column names (subset of global `cols`) for which we enumerate all training/new combinations.
    cols        : list of all considered columns
    training    : dict mapping column name to allowed TRAINING values
    new         : dict mapping column name to allowed NEW values

    Columns in `cols` but NOT in `cols_to_use`:
      - are constrained to values in TRAINING[col] ∪ NEW[col]
      - but are NOT shown as *_set columns in the output.
    """

    cols_to_use = list(cols_to_use)

    # Base mask: enforce "both" (TRAINING ∪ NEW) on the columns not selected in cols
    base_mask = pd.Series(True, index=meta_df.index)
    ignored_cols = [c for c in cols if c not in cols_to_use]

    for col in ignored_cols:
        allowed_values = set(training[col]) | set(new[col])
        base_mask &= meta_df[col].isin(allowed_values)

    results = []

    # Combinations only over the selected columns
    for choice in itertools.product(["training", "new"], repeat=len(cols_to_use)):
        mask = base_mask.copy()
        row_dict = {}

        for col, ch in zip(cols_to_use, choice):
            allowed_values = training[col] if ch == "training" else new[col]
            mask &= meta_df[col].isin(allowed_values)
            row_dict[f"{col}_set"] = ch

        row_dict["count"] = meta_df.loc[mask, "slide_id"].count()
        results.append(row_dict)

    summary_df = pd.DataFrame(results)

    # Number of NEW sets in the chosen columns
    if cols_to_use:
        summary_df["n_new"] = summary_df[[f"{c}_set" for c in cols_to_use]].apply(
            lambda r: sum(v == "new" for v in r),
            axis=1,
        )
    else:
        summary_df["n_new"] = 0

    # Sort by (fewest NEW first, then highest counts)
    summary_df = summary_df.sort_values(["n_new", "count"], ascending=[True, False])

    # Non-zero combinations only
    summary_nonzero = summary_df[summary_df["count"] > 0].copy()

    # Print total samples count
    print("Total samples (non-zero combinations):", summary_nonzero["count"].sum())

    return summary_df, summary_nonzero

def color_by_n_new(row):
    color = n_new_colors.get(row["n_new"], "#ffffff")
    return [f"background-color: {color}"] * len(row)

def highlight_new(val):
    if isinstance(val, str) and val.lower() == "new":
        return "color: red"
    return ""

In [4]:
def extend_meta_eval(meta_eval: pd.DataFrame,
                     test_set_info: pd.DataFrame,
                     magnification) -> pd.DataFrame:
    """
    Extend meta_eval with one row per unique slide_id from test_set_info.

    Parameters
    ----------
    meta_eval : pd.DataFrame
        Original metadata dataframe to extend.
    test_set_info : pd.DataFrame
        Dataframe containing at least ['slide_id', 'slide_set_scope', 'tissue_HistoST'].
    magnification :
        Value to put in the 'magnification' column for new rows (e.g. '40x').

    Returns
    -------
    pd.DataFrame
        New meta_eval with the additional rows appended.
    """

    # 1. Get unique slide_id with their scope and tissue
    slides = (
        test_set_info[['slide_id', 'slide_set_scope', 'tissue_HistoST']]
        .drop_duplicates(subset='slide_id')
        .copy()
    )

    # 2. Raise an error if any slide_id already exists in meta_eval
    existing_ids = set(meta_eval['slide_id'])
    duplicate_ids = set(slides['slide_id']) & existing_ids
    if duplicate_ids:
        raise ValueError(f"The following slide_id(s) already exist in meta_eval: {duplicate_ids}")

    # 3. Build new rows with the requested values
    new_rows = pd.DataFrame({
        'slide_id': slides['slide_id'],
        'tissue_HistoST': slides['tissue_HistoST'],
        'species': 'Homo sapiens',
        'st_technology': 'Xenium',
        'preservation_method': 'FFPE',
        'magnification': magnification,
        'source': 'STHELAR',
        # True if "shared", False if "specific"
        'see_in_training': slides['slide_set_scope'].eq('shared'),
        # subset is "STHELAR_train" if "shared" in "see_in_training", and "STHELAR_test" otherwise
        'subset': slides['slide_set_scope'].apply(lambda x: "STHELAR_train" if x == "shared" else "STHELAR_test")
    })

    # 4. Align to meta_eval columns, filling missing ones with NaN
    new_rows = new_rows.reindex(columns=meta_eval.columns)

    # 5. Concatenate and return
    meta_eval_extended = pd.concat([meta_eval, new_rows], ignore_index=True)

    return meta_eval_extended

## 1. Data from HEST

### Analyze data

In [5]:
meta_df = pd.read_csv("hf://datasets/MahmoodLab/hest/HEST_v1_3_0.csv")
meta_df

$HOME/miniforge3/envs/process_slide/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,dataset_title,id,image_filename,organ,disease_state,oncotree_code,species,patient,st_technology,data_publication_date,...,treatment_comment,pixel_size_um_embedded,pixel_size_um_estimated,magnification,fullres_px_width,fullres_px_height,tissue,disease_comment,subseries,hest_version_added
0,Xenium v1 Human Breast FFPE with Biomarkers & ...,TENX202,TENX202.tif,Breast,Cancer,IDC,Homo sapiens,Patient 4,Xenium,2025-12-11,...,NaN,0.273809,0.273968,NaN,21607,30322,Breast,"DCIS, T4a N2a, G3, HER2-1+","Section 4, bottom",v1_3_0
1,Xenium v1 Human Breast FFPE with Biomarkers & ...,TENX201,TENX201.tif,Breast,Cancer,IDC,Homo sapiens,Patient 4,Xenium,2025-12-11,...,NaN,0.273810,0.274040,NaN,22892,39456,Breast,"IDC, T2 N0 MX, Not provided, ER+/PR+/HER2-2+","Section 4, middle",v1_3_0
2,Xenium v1 Human Breast FFPE with Biomarkers & ...,TENX200,TENX200.tif,Breast,Cancer,IDC,Homo sapiens,Patient 4,Xenium,2025-12-11,...,NaN,0.273813,0.273959,NaN,26717,46370,Breast,"IDC, Tis NX MX, G3, ER+/PR+/HER2-1+","Section 4, top",v1_3_0
3,Xenium v1 Human Breast FFPE with Biomarkers & ...,TENX199,TENX199.tif,Breast,Cancer,IDC,Homo sapiens,Patient 3,Xenium,2025-12-11,...,NaN,0.273808,0.274062,NaN,14853,31607,Breast,"DCIS, TX NX M0, G2, ER-/PR-/HER2-2+","Section 3, bottom",v1_3_0
4,Xenium v1 Human Breast FFPE with Biomarkers & ...,TENX198,TENX198.tif,Breast,Cancer,IDC,Homo sapiens,Patient 3,Xenium,2025-12-11,...,NaN,0.273813,0.274110,NaN,32654,35502,Breast,"DCIS, T4b N3a M1, G2, ER+/PR+/HER2-2+","Section 3, middle",v1_3_0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1271,spatialLIBD,MISC5,MISC5.tif,Brain,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,NaN,0.727113,20x,13332,13332,dorsolateral prefrontal cortex,NaN,151672,v1_0_0
1272,spatialLIBD,MISC4,MISC4.tif,Brain,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,NaN,0.726109,20x,13332,13332,dorsolateral prefrontal cortex,NaN,151673,v1_0_0
1273,spatialLIBD,MISC3,MISC3.tif,Brain,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,NaN,0.725124,20x,13332,13332,dorsolateral prefrontal cortex,NaN,151674,v1_0_0
1274,spatialLIBD,MISC2,MISC2.tif,Brain,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,NaN,0.726109,20x,13332,13332,dorsolateral prefrontal cortex,NaN,151675,v1_0_0


In [6]:
meta_df.columns

Index(['dataset_title', 'id', 'image_filename', 'organ', 'disease_state',
       'oncotree_code', 'species', 'patient', 'st_technology',
       'data_publication_date', 'license', 'study_link', 'download_page_link1',
       'inter_spot_dist', 'spot_diameter', 'spots_under_tissue',
       'preservation_method', 'nb_genes', 'treatment_comment',
       'pixel_size_um_embedded', 'pixel_size_um_estimated', 'magnification',
       'fullres_px_width', 'fullres_px_height', 'tissue', 'disease_comment',
       'subseries', 'hest_version_added'],
      dtype='object')

In [7]:
# Rename 'id' to 'slide_id'
meta_df = meta_df.rename(columns={'id': 'slide_id'})

In [8]:
for v, c in meta_df.organ.value_counts(dropna=False).items():
    print(f"{v}: {c}")

Spinal cord: 318
Brain: 214
Breast: 140
Bowel: 95
Skin: 88
Lung: 72
Kidney: 72
Heart: 70
Prostate: 63
Liver: 37
Uterus: 24
Eye: 10
Muscle: 10
Bone: 10
Ovary: 10
Pancreas: 9
Lymphoid: 8
Bladder: 6
Cervix: 6
Lymph node: 5
Embryo: 3
Lung/Brain: 2
Whole organism: 1
Kidney/Brain: 1
Placenta: 1
Other: 1


In [9]:
# Check if some organs could be present in training by looking at the more precise tissue label
meta_df[meta_df.organ.isin(["Lymphoid", "Bowel"])].groupby(["organ", "tissue"]).size().reset_index(name="count")
# Needs refining for bowel (colon is present) and for lymphoid (lymph node, tonsil and bone marrow are present)

,organ,tissue,count
0,Bowel,Bowel,1
1,Bowel,Cecum,2
2,Bowel,Colon,63
3,Bowel,Colon (Sigma),2
4,Bowel,Colon (right),4
5,Bowel,Colorectal,1
6,Bowel,Intestine,1
7,Bowel,Large intestine,2
8,Bowel,Rectum,4
9,Bowel,Sigma/Rectum,2


In [10]:
def title_no_space(s):
    """Capitalize each word, remove spaces."""
    if pd.isna(s):
        return s
    return ''.join(word.capitalize() for word in s.split())

def resolve_lymphoid(tissue):
    """Specific rules when organ == 'Lymphoid'."""
    if pd.isna(tissue):
        return None
    t = tissue.strip().lower()
    if t == "bone marrow":
        return "BoneMarrow"
    if t == "lymph node (left axilla)" or t == "prelaryngeal lymph node	":
        return "LymphNode"
    if t == "tonsil":
        return "Tonsil"
    return None  # fallback if unmatched

def resolve_bowel(tissue):
    """Specific rules when organ == 'Bowel'."""
    if pd.isna(tissue):
        return "Bowel"
    t = tissue.lower()
    # colon / colorectal / intestine anywhere
    if re.search(r"colon|colorectal|intestin", t):
        return "Colon"
    return "Bowel"

# Build the new column
def compute_tissue(row):
    organ = row["organ"]
    tissue = row["tissue"]

    if organ == "Lymphoid":
        val = resolve_lymphoid(tissue)
        if val is not None:
            return val
        # fallback: apply generic formatting if needed
        return title_no_space(organ)

    if organ == "Bowel":
        return resolve_bowel(tissue)

    # Default case: formatted organ name
    return title_no_space(organ)

meta_df["tissue_HistoST"] = meta_df.apply(compute_tissue, axis=1)

In [11]:
meta_df.tissue_HistoST.value_counts(dropna=False)

tissue_HistoST
SpinalCord       318
Brain            214
Breast           140
Skin              88
Colon             78
Lung              72
Kidney            72
Heart             70
Prostate          63
Liver             37
Uterus            24
Bowel             17
Eye               10
Muscle            10
Bone              10
Ovary             10
Pancreas           9
LymphNode          6
Bladder            6
Cervix             6
Tonsil             4
Embryo             3
BoneMarrow         2
Lung/brain         2
WholeOrganism      1
Kidney/brain       1
Lymphoid           1
Placenta           1
Other              1
Name: count, dtype: int64

In [12]:
meta_df.species.value_counts(dropna=False)

species
Homo sapiens                 689
Mus musculus                 586
Homo sapiens/Mus musculus      1
Name: count, dtype: int64

In [13]:
meta_df.st_technology.value_counts(dropna=False)

st_technology
Visium                     602
Spatial Transcriptomics    552
Xenium                      79
Visium HD                   28
Visium HD 3'                 8
Xenium 5k                    7
Name: count, dtype: int64

In [14]:
meta_df.preservation_method.value_counts(dropna=False)

preservation_method
NaN             563
Fresh Frozen    458
FFPE            214
Snap Frozen      19
PFA fixed         7
PFA               7
Fixed Frozen      5
Flash Frozen      3
Name: count, dtype: int64

In [15]:
meta_df.magnification.value_counts(dropna=False)

magnification
20x    787
40x    451
NaN     21
10x     17
Name: count, dtype: int64

In [16]:
# Value counts for technology where preservation method is NA
meta_df[meta_df["preservation_method"].isna()].st_technology.value_counts(dropna=False)

st_technology
Spatial Transcriptomics    461
Visium                     102
Name: count, dtype: int64

In [17]:
# Data where magnification is 10x (technology, preservation method, tissue, and species columns)
meta_df[meta_df.magnification == "10x"][["st_technology", "preservation_method", "tissue_HistoST", "species"]].value_counts(dropna=False).reset_index(name="count")

,st_technology,preservation_method,tissue_HistoST,species,count
0,Visium,Fresh Frozen,Skin,Homo sapiens,9
1,Visium,Fresh Frozen,Brain,Mus musculus,6
2,Visium,FFPE,Eye,Homo sapiens,1
3,Visium,Fresh Frozen,Other,Mus musculus,1


In [18]:
TRAINING_TISSUES = ["Breast", "Skin", "Colon", "Bowel", "Heart", "Kidney", "Prostate", "Lung", "Liver", "Pancreas", "LymphNode", "Cervix", "Uterus", "Ovary", "Tonsil"]
NEW_TISSUES = ["SpinalCord", "Bone", "Eye", "Muscle", "Bladder", "Embryo", "BoneMarrow", "Placenta"] # Do not include Brain, too specific tissue

TRAINING_SPECIES = ["Homo sapiens"]
NEW_SPECIES = ["Mus musculus"]

TRAINING_TECHNOLOGIES = ["Xenium", "Xenium 5k"]   # HEST Xenium slides are not used (avoids leakage with the training data); the internal Xenium test set is used instead
NEW_TECHNOLOGIES = ["Spatial Transcriptomics", "Visium", "Visium HD", "Visium HD 3'"]

TRAINING_PRESERVATION = ["FFPE"]
NEW_PRESERVATION = ["Fresh Frozen", "Snap Frozen", "Fixed Frozen", "Flash Frozen"]

TRAINING_MAGNIFICATION = ["20x"]
NEW_MAGNIFICATION = ["40x"]

In [19]:
cols = ["tissue_HistoST", "species", "st_technology", "preservation_method", "magnification"]

training = {
    "tissue_HistoST": TRAINING_TISSUES,
    "species": TRAINING_SPECIES,
    "st_technology": TRAINING_TECHNOLOGIES,
    "preservation_method": TRAINING_PRESERVATION,
    "magnification": TRAINING_MAGNIFICATION,
}

new = {
    "tissue_HistoST": NEW_TISSUES,
    "species": NEW_SPECIES,
    "st_technology": NEW_TECHNOLOGIES,
    "preservation_method": NEW_PRESERVATION,
    "magnification": NEW_MAGNIFICATION,
}

summary_df, summary_nonzero = summarize_combinations(meta_df, cols_to_use=cols, cols=cols, training=training, new=new)
styled_nonzero = (
    summary_nonzero
        .style
        .apply(color_by_n_new, axis=1)
        .applymap(highlight_new)
)

styled_nonzero

Total samples (non-zero combinations): 552


/var/folders/ff/llzyqd_s78v1k8vywvjxp5_r0000gn/T/ipykernel_37871/1831928480.py:24: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_new)


,tissue_HistoST_set,species_set,st_technology_set,preservation_method_set,magnification_set,count,n_new
1,training,training,training,training,new,55,1
4,training,training,new,training,training,16,1
6,training,training,new,new,training,180,2
5,training,training,new,training,new,78,2
17,new,training,training,training,new,3,2
20,new,training,new,training,training,2,2
7,training,training,new,new,new,141,3
14,training,new,new,new,training,25,3
28,new,new,new,training,training,4,3
13,training,new,new,training,new,3,3


### Select data to download

In [20]:
# Test other technologies
ids_other_techno = meta_df[meta_df.tissue_HistoST.isin(TRAINING_TISSUES)
                        & meta_df.species.isin(TRAINING_SPECIES)
                        & meta_df.st_technology.isin(NEW_TECHNOLOGIES)
                        & meta_df.preservation_method.isin(TRAINING_PRESERVATION)
                        & meta_df.magnification.isin(TRAINING_MAGNIFICATION)].slide_id
ids_other_techno.count()

np.int64(16)

In [21]:
meta_df[meta_df.slide_id.isin(ids_other_techno.tolist())].tissue_HistoST.value_counts()

tissue_HistoST
Skin      12
Ovary      1
Cervix     1
Colon      1
Lung       1
Name: count, dtype: int64

In [22]:
# Test other magnifications (cannot use TRAINING_TECHNOLOGIES (Xenium) to avoid data leakage)
ids_other_magnification = meta_df[meta_df.tissue_HistoST.isin(TRAINING_TISSUES)
                                & meta_df.species.isin(TRAINING_SPECIES)
                                & meta_df.st_technology.isin(NEW_TECHNOLOGIES)
                                & meta_df.preservation_method.isin(TRAINING_PRESERVATION)
                                & meta_df.magnification.isin(NEW_MAGNIFICATION)].slide_id
ids_other_magnification.count()

np.int64(78)

In [23]:
meta_df[meta_df.slide_id.isin(ids_other_magnification.tolist())].tissue_HistoST.value_counts()

tissue_HistoST
Colon       52
Lung        10
Prostate     4
Skin         4
Heart        3
Kidney       2
Pancreas     1
Ovary        1
Breast       1
Name: count, dtype: int64

In [24]:
# Test other organs
ids_other_organs = meta_df[meta_df.tissue_HistoST.isin(NEW_TISSUES)
                        & meta_df.species.isin(TRAINING_SPECIES)
                        & meta_df.st_technology.isin(NEW_TECHNOLOGIES)
                        & meta_df.preservation_method.isin(TRAINING_PRESERVATION)
                        & meta_df.magnification.isin(TRAINING_MAGNIFICATION)].slide_id
ids_other_organs.count()

np.int64(2)

In [25]:
meta_df[meta_df.slide_id.isin(ids_other_organs.tolist())].tissue_HistoST.value_counts()

tissue_HistoST
Bladder    2
Name: count, dtype: int64

In [26]:
# Test other preservation methods
ids_other_preservation = meta_df[meta_df.tissue_HistoST.isin(TRAINING_TISSUES)
                                & meta_df.species.isin(TRAINING_SPECIES)
                                & meta_df.st_technology.isin(NEW_TECHNOLOGIES)
                                & meta_df.preservation_method.isin(NEW_PRESERVATION)
                                & meta_df.magnification.isin(TRAINING_MAGNIFICATION)].slide_id
ids_other_preservation.count()

np.int64(180)

In [27]:
meta_df[meta_df.slide_id.isin(ids_other_preservation.tolist())].tissue_HistoST.value_counts()

tissue_HistoST
Skin         46
Breast       37
Heart        30
Kidney       30
Uterus       15
Liver         8
Prostate      7
Pancreas      4
Colon         2
LymphNode     1
Name: count, dtype: int64

In [28]:
# Test other species (only available for new preservation methods)
ids_other_species = meta_df[meta_df.tissue_HistoST.isin(TRAINING_TISSUES)
                          & meta_df.species.isin(NEW_SPECIES)
                          & meta_df.st_technology.isin(NEW_TECHNOLOGIES)
                          & meta_df.preservation_method.isin(NEW_PRESERVATION)
                          & meta_df.magnification.isin(TRAINING_MAGNIFICATION)].slide_id
ids_other_species.count()

np.int64(25)

In [29]:
meta_df[meta_df.slide_id.isin(ids_other_species.tolist())].tissue_HistoST.value_counts()

tissue_HistoST
Liver       11
Uterus       7
Heart        4
Prostate     2
Kidney       1
Name: count, dtype: int64

In [30]:
ids_to_download = pd.concat([ids_other_techno,
                             ids_other_magnification,
                             ids_other_organs,
                             ids_other_preservation,
                             ids_other_species])
# Check there are no duplicates
print("Number of duplicates in ids_to_download:", ids_to_download.duplicated().sum())
print("Total unique ids to download:", ids_to_download.nunique())

Number of duplicates in ids_to_download: 0
Total unique ids to download: 301


In [31]:
# Filter meta_df to only include ids in ids_to_download
meta_to_download = meta_df[meta_df.slide_id.isin(ids_to_download)]
# Add a column indicating data coming from HEST and slide never seen during training
meta_to_download["source"] = "HEST"
meta_to_download["see_in_training"] = False
meta_to_download

/var/folders/ff/llzyqd_s78v1k8vywvjxp5_r0000gn/T/ipykernel_37871/1375086599.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_to_download["source"] = "HEST"
/var/folders/ff/llzyqd_s78v1k8vywvjxp5_r0000gn/T/ipykernel_37871/1375086599.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_to_download["see_in_training"] = False


,dataset_title,slide_id,image_filename,organ,disease_state,oncotree_code,species,patient,st_technology,data_publication_date,...,magnification,fullres_px_width,fullres_px_height,tissue,disease_comment,subseries,hest_version_added,tissue_HistoST,source,see_in_training
33,"Visium HD Spatial Gene Expression Library, Hum...",TENX174,TENX174.tif,Prostate,Cancer,PRAD,Homo sapiens,NaN,Visium HD,2025-06-10,...,40x,28288,24794,Prostate,NaN,NaN,v1_2_0,Prostate,HEST,False
34,"Visium HD Spatial Gene Expression Library, Hum...",TENX173,TENX173.tif,Kidney,Healthy,NaN,Homo sapiens,NaN,Visium HD,2025-03-06,...,40x,32674,43999,Kidney,NaN,NaN,v1_2_0,Kidney,HEST,False
36,"Visium HD Spatial Gene Expression Libraries, P...",TENX171,TENX171.tif,Lung,Cancer,LUAD,Homo sapiens,NaN,Visium HD,2024-09-24,...,40x,26720,43270,Lung,NaN,"Experiment 2, post-Xenium Prime 5K",v1_2_0,Lung,HEST,False
37,"Visium HD Spatial Gene Expression Libraries, P...",TENX170,TENX170.tif,Lung,Cancer,LUAD,Homo sapiens,NaN,Visium HD,2024-09-24,...,40x,25078,43266,Lung,NaN,"Experiment 2, HD only (control)",v1_2_0,Lung,HEST,False
38,"Visium HD Spatial Gene Expression Libraries, P...",TENX169,TENX169.tif,Lung,Cancer,LUAD,Homo sapiens,NaN,Visium HD,2024-09-24,...,40x,25078,43448,Lung,NaN,"Experiment 1, post-Xenium v1",v1_2_0,Lung,HEST,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
833,Spatial transcriptomics landscape of non-commu...,NCBI469,NCBI469.tif,Skin,Diseased,NaN,Homo sapiens,NaN,Visium,14-Nov-22,...,20x,15983,16190,Skin,atopic dermatitis lesional,"Patient 15 lesional atopic dermatitis, rep 2 [...",v1_0_0,Skin,HEST,False
1260,A spatially resolved atlas of the human lung c...,MISC16,MISC16.tif,Lung,Treated,NaN,Homo sapiens,,Visium,21-Dec-22,...,40x,33264,18688,Bronchi,NaN,WSA_LngSP10972072,v1_0_0,Lung,HEST,False
1261,A spatially resolved atlas of the human lung c...,MISC15,MISC15.tif,Lung,Treated,NaN,Homo sapiens,,Visium,21-Dec-22,...,40x,33264,18688,Bronchi,NaN,WSA_LngSP10972073,v1_0_0,Lung,HEST,False
1262,A spatially resolved atlas of the human lung c...,MISC14,MISC14.tif,Lung,Treated,NaN,Homo sapiens,,Visium,21-Dec-22,...,40x,33264,18688,Trachea,NaN,WSA_LngSP10972074,v1_0_0,Lung,HEST,False


In [32]:
# New column to know in which set the slide was used
def assign_set(row):
    if row["slide_id"] in ids_other_techno.tolist():
        return "other_technologies"
    if row["slide_id"] in ids_other_magnification.tolist():
        return "other_magnifications"
    if row["slide_id"] in ids_other_organs.tolist():
        return "other_organs"
    if row["slide_id"] in ids_other_preservation.tolist():
        return "other_preservations"
    if row["slide_id"] in ids_other_species.tolist():
        return "other_species"
    return "unknown"
meta_to_download["subset"] = meta_to_download.apply(assign_set, axis=1)
meta_to_download

/var/folders/ff/llzyqd_s78v1k8vywvjxp5_r0000gn/T/ipykernel_37871/3535938681.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_to_download["subset"] = meta_to_download.apply(assign_set, axis=1)


,dataset_title,slide_id,image_filename,organ,disease_state,oncotree_code,species,patient,st_technology,data_publication_date,...,fullres_px_width,fullres_px_height,tissue,disease_comment,subseries,hest_version_added,tissue_HistoST,source,see_in_training,subset
33,"Visium HD Spatial Gene Expression Library, Hum...",TENX174,TENX174.tif,Prostate,Cancer,PRAD,Homo sapiens,NaN,Visium HD,2025-06-10,...,28288,24794,Prostate,NaN,NaN,v1_2_0,Prostate,HEST,False,other_magnifications
34,"Visium HD Spatial Gene Expression Library, Hum...",TENX173,TENX173.tif,Kidney,Healthy,NaN,Homo sapiens,NaN,Visium HD,2025-03-06,...,32674,43999,Kidney,NaN,NaN,v1_2_0,Kidney,HEST,False,other_magnifications
36,"Visium HD Spatial Gene Expression Libraries, P...",TENX171,TENX171.tif,Lung,Cancer,LUAD,Homo sapiens,NaN,Visium HD,2024-09-24,...,26720,43270,Lung,NaN,"Experiment 2, post-Xenium Prime 5K",v1_2_0,Lung,HEST,False,other_magnifications
37,"Visium HD Spatial Gene Expression Libraries, P...",TENX170,TENX170.tif,Lung,Cancer,LUAD,Homo sapiens,NaN,Visium HD,2024-09-24,...,25078,43266,Lung,NaN,"Experiment 2, HD only (control)",v1_2_0,Lung,HEST,False,other_magnifications
38,"Visium HD Spatial Gene Expression Libraries, P...",TENX169,TENX169.tif,Lung,Cancer,LUAD,Homo sapiens,NaN,Visium HD,2024-09-24,...,25078,43448,Lung,NaN,"Experiment 1, post-Xenium v1",v1_2_0,Lung,HEST,False,other_magnifications
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
833,Spatial transcriptomics landscape of non-commu...,NCBI469,NCBI469.tif,Skin,Diseased,NaN,Homo sapiens,NaN,Visium,14-Nov-22,...,15983,16190,Skin,atopic dermatitis lesional,"Patient 15 lesional atopic dermatitis, rep 2 [...",v1_0_0,Skin,HEST,False,other_preservations
1260,A spatially resolved atlas of the human lung c...,MISC16,MISC16.tif,Lung,Treated,NaN,Homo sapiens,,Visium,21-Dec-22,...,33264,18688,Bronchi,NaN,WSA_LngSP10972072,v1_0_0,Lung,HEST,False,other_magnifications
1261,A spatially resolved atlas of the human lung c...,MISC15,MISC15.tif,Lung,Treated,NaN,Homo sapiens,,Visium,21-Dec-22,...,33264,18688,Bronchi,NaN,WSA_LngSP10972073,v1_0_0,Lung,HEST,False,other_magnifications
1262,A spatially resolved atlas of the human lung c...,MISC14,MISC14.tif,Lung,Treated,NaN,Homo sapiens,,Visium,21-Dec-22,...,33264,18688,Trachea,NaN,WSA_LngSP10972074,v1_0_0,Lung,HEST,False,other_magnifications


## 2. Internal test set (Xenium)

In [33]:
# Load saved HEST metadata
meta_eval = meta_to_download.copy()

In [34]:
print(meta_eval.iloc[0])

dataset_title              Visium HD Spatial Gene Expression Library, Hum...
slide_id                                                             TENX174
image_filename                                                   TENX174.tif
organ                                                               Prostate
disease_state                                                         Cancer
oncotree_code                                                           PRAD
species                                                         Homo sapiens
patient                                                                  NaN
st_technology                                                      Visium HD
data_publication_date                                             2025-06-10
license                                         Creative Commons Attribution
study_link                                                               NaN
download_page_link1        https://www.10xgenomics.com/datasets/visium-hd...

In [35]:
# Load test set information
test_set_info = pd.read_csv(get_full_path('train', 'training_datasets', dataset_id, training_dataset_id, 'cell_count_test.csv', as_str=True))
test_set_info["slide_id"] = (test_set_info["patch_id"].str.split("_").str[:-1].str.join("_"))
test_set_info["tissue_HistoST"] = (test_set_info["slide_id"].str.split("_").str[:-1].str.join(" ")).apply(title_no_space)
test_set_info

,patch_id,T_NK,B_Plasma,Myeloid,Blood_vessel,Fibroblast_Myofibroblast,Epithelial,Specialized,Melanocyte,Unknown,slide_set_scope,slide_id,tissue_HistoST
0,breast_s0_43870,0,0,0,0,0,5,0,0,0,shared,breast_s0,Breast
1,breast_s0_43871,0,0,0,0,0,9,0,0,0,shared,breast_s0,Breast
2,breast_s0_43872,0,0,1,0,0,6,3,0,0,shared,breast_s0,Breast
3,breast_s0_43873,1,0,5,12,4,2,7,0,0,shared,breast_s0,Breast
4,breast_s0_43874,2,0,8,13,12,20,10,0,0,shared,breast_s0,Breast
...,...,...,...,...,...,...,...,...,...,...,...,...,...
54101,cervix_s0_34044,0,0,0,0,5,0,0,0,13,shared,cervix_s0,Cervix
54102,cervix_s0_34045,2,0,0,0,8,0,0,0,17,shared,cervix_s0,Cervix
54103,cervix_s0_34046,2,0,0,0,4,0,0,0,9,shared,cervix_s0,Cervix
54104,cervix_s0_34047,0,0,0,0,1,1,0,0,4,shared,cervix_s0,Cervix


In [36]:
test_set_info.slide_id.unique()

array(['breast_s0', 'breast_s1', 'breast_s6', 'lung_s1', 'lung_s3',
       'skin_s1', 'skin_s2', 'skin_s3', 'skin_s4', 'pancreatic_s0',
       'pancreatic_s1', 'pancreatic_s2', 'heart_s0', 'colon_s1',
       'colon_s2', 'kidney_s0', 'kidney_s1', 'kidney_s2', 'liver_s0',
       'liver_s1', 'tonsil_s0', 'tonsil_s1', 'lymph_node_s0', 'ovary_s0',
       'ovary_s1', 'prostate_s0', 'cervix_s0'], dtype=object)

In [37]:
test_set_info.tissue_HistoST.value_counts(dropna=False)

tissue_HistoST
Breast        17419
Pancreatic     8945
Kidney         8032
Lung           3188
Cervix         2916
Tonsil         2573
Skin           2560
Ovary          2406
Liver          2233
Colon          1873
Prostate        926
LymphNode       670
Heart           365
Name: count, dtype: int64

In [38]:
meta_eval = extend_meta_eval(meta_eval, test_set_info, magnification='20x')
meta_eval.iloc[-5:]

,dataset_title,slide_id,image_filename,organ,disease_state,oncotree_code,species,patient,st_technology,data_publication_date,...,fullres_px_width,fullres_px_height,tissue,disease_comment,subseries,hest_version_added,tissue_HistoST,source,see_in_training,subset
323,NaN,lymph_node_s0,NaN,NaN,NaN,NaN,Homo sapiens,NaN,Xenium,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,LymphNode,STHELAR,True,STHELAR_train
324,NaN,ovary_s0,NaN,NaN,NaN,NaN,Homo sapiens,NaN,Xenium,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Ovary,STHELAR,True,STHELAR_train
325,NaN,ovary_s1,NaN,NaN,NaN,NaN,Homo sapiens,NaN,Xenium,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Ovary,STHELAR,True,STHELAR_train
326,NaN,prostate_s0,NaN,NaN,NaN,NaN,Homo sapiens,NaN,Xenium,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Prostate,STHELAR,True,STHELAR_train
327,NaN,cervix_s0,NaN,NaN,NaN,NaN,Homo sapiens,NaN,Xenium,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Cervix,STHELAR,True,STHELAR_train


In [39]:
# slide count where subset is STHELAR_train
print("STHELAR_train slide count:", meta_eval[meta_eval.subset == "STHELAR_train"].shape[0])
# slide count where subset is STHELAR_test
print("STHELAR_test slide count:", meta_eval[meta_eval.subset == "STHELAR_test"].shape[0])

STHELAR_train slide count: 25
STHELAR_test slide count: 2


In [40]:
# Number of patch ids in each tissue in STHELAR_train
meta_eval[meta_eval.subset == "STHELAR_train"].groupby("tissue_HistoST").size()

tissue_HistoST
Breast        3
Cervix        1
Colon         2
Heart         1
Kidney        2
Liver         2
Lung          2
LymphNode     1
Ovary         2
Pancreatic    2
Prostate      1
Skin          4
Tonsil        2
dtype: int64

In [41]:
# Number of patch ids in each tissue in STHELAR_test
meta_eval[meta_eval.subset == "STHELAR_test"].groupby("tissue_HistoST").size()

tissue_HistoST
Kidney        1
Pancreatic    1
dtype: int64

In [42]:
# Number of patch ids in each slide_id in STHELAR_train using test_set_info
test_set_info[test_set_info.slide_id.isin(
    meta_eval[meta_eval.subset == "STHELAR_train"].slide_id.tolist()
)].groupby("slide_id").size()

slide_id
breast_s0        4225
breast_s1        9161
breast_s6        4033
cervix_s0        2916
colon_s1         1133
colon_s2          740
heart_s0          365
kidney_s1         507
kidney_s2        2179
liver_s0         1576
liver_s1          657
lung_s1          1142
lung_s3          2046
lymph_node_s0     670
ovary_s0          566
ovary_s1         1840
pancreatic_s0    2074
pancreatic_s1    1564
prostate_s0       926
skin_s1           850
skin_s2           604
skin_s3           205
skin_s4           901
tonsil_s0        1020
tonsil_s1        1553
dtype: int64

In [43]:
# Number of patch ids in each slide_id in STHELAR_test using test_set_info
test_set_info[test_set_info.slide_id.isin(
    meta_eval[meta_eval.subset == "STHELAR_test"].slide_id.tolist()
)].groupby("slide_id").size()

slide_id
kidney_s0        5346
pancreatic_s2    5307
dtype: int64

## 3. Save csv metadata of samples to download

In [44]:
# Save metadata of samples to download
path = get_full_path('eval', 'eval_datasets', 'st_eval_data', f'eval_st_{training_dataset_id}', f'metadata_eval_st_{training_dataset_id}.csv', as_str=True)
os.makedirs(os.path.dirname(path), exist_ok=True)
meta_eval.to_csv(path, index=False)

In [45]:
meta_eval

,dataset_title,slide_id,image_filename,organ,disease_state,oncotree_code,species,patient,st_technology,data_publication_date,...,fullres_px_width,fullres_px_height,tissue,disease_comment,subseries,hest_version_added,tissue_HistoST,source,see_in_training,subset
0,"Visium HD Spatial Gene Expression Library, Hum...",TENX174,TENX174.tif,Prostate,Cancer,PRAD,Homo sapiens,NaN,Visium HD,2025-06-10,...,28288.0,24794.0,Prostate,NaN,NaN,v1_2_0,Prostate,HEST,False,other_magnifications
1,"Visium HD Spatial Gene Expression Library, Hum...",TENX173,TENX173.tif,Kidney,Healthy,NaN,Homo sapiens,NaN,Visium HD,2025-03-06,...,32674.0,43999.0,Kidney,NaN,NaN,v1_2_0,Kidney,HEST,False,other_magnifications
2,"Visium HD Spatial Gene Expression Libraries, P...",TENX171,TENX171.tif,Lung,Cancer,LUAD,Homo sapiens,NaN,Visium HD,2024-09-24,...,26720.0,43270.0,Lung,NaN,"Experiment 2, post-Xenium Prime 5K",v1_2_0,Lung,HEST,False,other_magnifications
3,"Visium HD Spatial Gene Expression Libraries, P...",TENX170,TENX170.tif,Lung,Cancer,LUAD,Homo sapiens,NaN,Visium HD,2024-09-24,...,25078.0,43266.0,Lung,NaN,"Experiment 2, HD only (control)",v1_2_0,Lung,HEST,False,other_magnifications
4,"Visium HD Spatial Gene Expression Libraries, P...",TENX169,TENX169.tif,Lung,Cancer,LUAD,Homo sapiens,NaN,Visium HD,2024-09-24,...,25078.0,43448.0,Lung,NaN,"Experiment 1, post-Xenium v1",v1_2_0,Lung,HEST,False,other_magnifications
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
323,NaN,lymph_node_s0,NaN,NaN,NaN,NaN,Homo sapiens,NaN,Xenium,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,LymphNode,STHELAR,True,STHELAR_train
324,NaN,ovary_s0,NaN,NaN,NaN,NaN,Homo sapiens,NaN,Xenium,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Ovary,STHELAR,True,STHELAR_train
325,NaN,ovary_s1,NaN,NaN,NaN,NaN,Homo sapiens,NaN,Xenium,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Ovary,STHELAR,True,STHELAR_train
326,NaN,prostate_s0,NaN,NaN,NaN,NaN,Homo sapiens,NaN,Xenium,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Prostate,STHELAR,True,STHELAR_train
